In [2]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path

# === Config fissi ===
TSLAON = "0xf6b1117ec07684d3958cad8beb1b302bfd21103f"
USDC   = "0xA0b86991c6218b36c1d19d4a2e9eb0ce3606eb48"

CSV_ALL = "erc20_transfers_in_tslaon_txs_last7d_raw.csv"
OUT_PRICES = "tslaon_usdc_prices.csv"
OUT_DIAG   = "tslaon_usdc_diagnostics.csv"

# Router/aggregatori da ignorare
KNOWN_INFRA = {
    "0x7a250d5630b4cf539739df2c5dacb4c659f2488d",  # UniswapV2Router
    "0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b",  # UniversalRouter
    "0x1111111254fb6c44bac0bed2854e76f90643097d",  # 1inch
    "0x11111112542d85b3ef69ae05771c2dccff4faa26",
}

df = pd.read_csv(CSV_ALL)
for col in ["contract_address","from_address","to_address"]:
    df[col] = df[col].astype(str).str.lower()

tslaon = TSLAON.lower()
usdc   = USDC.lower()
dfp = df[df["contract_address"].isin([tslaon, usdc])].copy()

if "value_decimal" not in dfp.columns and {"value_raw","decimals"}.issubset(dfp.columns):
    dfp["value_decimal"] = dfp["value_raw"] / (10 ** dfp["decimals"].astype(int))

# Costruisci flussi firmati
signed = []
for _, r in dfp.iterrows():
    amt = float(r.get("value_decimal", 0.0) or 0.0)
    txh = r["tx_hash"]; ts = r.get("block_timestamp")
    tok = r["contract_address"]
    signed.append({"tx_hash": txh,"block_timestamp": ts,"address": r["from_address"],"token": tok,"signed_amount": -amt})
    signed.append({"tx_hash": txh,"block_timestamp": ts,"address": r["to_address"],"token": tok,"signed_amount": +amt})
flow = pd.DataFrame(signed)

# Pivot per (tx, address)
piv = flow.pivot_table(index=["tx_hash","block_timestamp","address"],
                       columns="token", values="signed_amount",
                       aggfunc="sum", fill_value=0.0).reset_index()
piv = piv.rename(columns={tslaon:"net_tslaon", usdc:"net_usdc"})
if "net_tslaon" not in piv: piv["net_tslaon"]=0.0
if "net_usdc" not in piv:   piv["net_usdc"]=0.0
piv.to_csv(OUT_DIAG, index=False)

results=[]
ignored=set(KNOWN_INFRA)

for tx, grp in piv.groupby("tx_hash"):
    cand_buy  = grp[(grp["net_usdc"]<0)&(grp["net_tslaon"]>0)]
    cand_sell = grp[(grp["net_usdc"]>0)&(grp["net_tslaon"]<0)]
    def pick(cands):
        c2=cands[~cands["address"].isin(ignored)]
        if c2.empty and cands.empty: return None
        base=c2 if not c2.empty else cands
        return base.reindex(base["net_tslaon"].abs().sort_values(ascending=False).index).iloc[0]
    chosen=None; side=None
    b=pick(cand_buy); s=pick(cand_sell)
    if b is not None and (s is None or abs(b["net_tslaon"])>=abs(s["net_tslaon"])): chosen,side=b,"buy"
    elif s is not None: chosen,side=s,"sell"
    if chosen is None: continue
    q_ts=abs(float(chosen["net_tslaon"])); q_us=abs(float(chosen["net_usdc"]))
    if q_ts>0 and q_us>0:
        results.append({
            "tx_hash":tx,"block_timestamp":chosen["block_timestamp"],
            "trader":chosen["address"],"side":side,
            "tslaon_qty":q_ts,"usdc_qty":q_us,
            "price_usdc_per_tslaon":q_us/q_ts
        })

prices=pd.DataFrame(results).sort_values("block_timestamp")
prices.to_csv(OUT_PRICES,index=False)
print(f"[✓] Scritto {OUT_PRICES} ({len(prices)} righe)")
print(f"[✓] Scritto {OUT_DIAG} ({len(piv)} righe diagnostica)")


[✓] Scritto tslaon_usdc_prices.csv (152 righe)
[✓] Scritto tslaon_usdc_diagnostics.csv (5462 righe diagnostica)


In [3]:
# === Coverage: percentuale di eventi TSLAon con prezzo ricostruibile ===
CSV_TSLAON = "tslaon_transfers_last7d_raw.csv"

try:
    df_ts = pd.read_csv(CSV_TSLAON)
except FileNotFoundError:
    print(f"[i] Copertura: non trovo {CSV_TSLAON}, salto il calcolo della percentuale.")
else:
    # normalizza indirizzi e prepara flag mint/burn
    for col in ["from_address","to_address"]:
        if col in df_ts.columns:
            df_ts[col] = df_ts[col].astype(str).str.lower()

    zero = "0x0000000000000000000000000000000000000000"
    df_ts["is_mint_burn"] = (df_ts["from_address"] == zero) | (df_ts["to_address"] == zero)

    # set di tx con prezzo ricostruito
    tx_with_price = set(prices["tx_hash"].astype(str)) if not prices.empty else set()

    # copertura su TUTTI gli eventi TSLAon
    total_events = len(df_ts)
    covered_events = df_ts["tx_hash"].astype(str).isin(tx_with_price).sum() if total_events else 0
    pct_all = (covered_events / total_events * 100.0) if total_events else 0.0

    # copertura su eventi "utili" (esclude mint/burn)
    df_useful = df_ts[~df_ts["is_mint_burn"]].copy()
    total_useful = len(df_useful)
    covered_useful = df_useful["tx_hash"].astype(str).isin(tx_with_price).sum() if total_useful else 0
    pct_useful = (covered_useful / total_useful * 100.0) if total_useful else 0.0

    print("\n[≣] Copertura ricostruzione prezzo")
    print(f"    - Eventi TSLAon totali:         {total_events}")
    print(f"    - Eventi TSLAon coperti:        {covered_events}  ({pct_all:.2f}%)")
    print(f"    - Eventi TSLAon utili (no 0x0): {total_useful}")
    print(f"    - Eventi utili coperti:         {covered_useful}  ({pct_useful:.2f}%)")



[≣] Copertura ricostruzione prezzo
    - Eventi TSLAon totali:         2034
    - Eventi TSLAon coperti:        341  (16.76%)
    - Eventi TSLAon utili (no 0x0): 1445
    - Eventi utili coperti:         274  (18.96%)
